# z2 — 30-minute Colab run (NaNROUTER diverse + Gutenberg)

**Runtime → Change runtime type → T4 GPU** before running.

This trains the 84.86M v2 hybrid for 30 minutes on a mixed corpus:

- `corpora-paragraphs-0010`
- `corpora-short-stories-0004`
- `corpora-sentences-0010`
- `hw5-gutenberg-c256-g1` (21-book Gutenberg reference)

Artifacts are saved to Google Drive, not ephemeral `/content`.


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tokenizers zstandard

In [ ]:
!test -d /content/NaNROUTER || git clone --depth 1 https://github.com/shemshallah/NaNROUTER /content/NaNROUTER
!test -d /content/nanlm_experimental || git clone --depth 1 https://github.com/shemshallah/nanlm_experimental /content/nanlm_experimental
!cd /content/NaNROUTER && git pull --rebase
!cd /content/nanlm_experimental && git pull --rebase


## Persist artifacts

Mount Drive. If you skip this, everything is lost when the Colab runtime ends.


In [ ]:
from google.colab import drive
import os, shutil, time

MOUNT = '/content/drive'
try:
    drive.mount(MOUNT, force_remount=True)
except ValueError as e:
    if os.path.isdir(f'{MOUNT}/MyDrive'):
        print('Drive already mounted at', MOUNT)
    else:
        # Move non-Drive leftovers aside instead of deleting them.
        backup = '/content/drive_pre_mount_backup'
        os.makedirs(backup, exist_ok=True)
        for name in os.listdir(MOUNT):
            src = os.path.join(MOUNT, name)
            dst = os.path.join(backup, name)
            if os.path.exists(dst):
                dst = f'{dst}.{int(time.time())}'
            shutil.move(src, dst)
        print('moved leftovers to', backup)
        drive.mount(MOUNT, force_remount=True)

OUT = f'{MOUNT}/MyDrive/nanlmv3_z2'
os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
print('OUT =', OUT)


## Run the 30-minute job

The script downloads the corpus releases, builds a mixed 200MB corpus, reuses
the z1 BPE tokenizer, and trains for 1,800 seconds with time-cosine LR to zero
and EMA weights.


In [ ]:
import os
CKPT = f"{OUT}/best.pt"
print('checkpoint:', CKPT)
print('exists:', os.path.exists(CKPT))
if os.path.exists(CKPT):
    print('size_mb:', round(os.path.getsize(CKPT) / 1e6, 1))
    print('mtime:', os.path.getmtime(CKPT))


In [ ]:
!python /content/nanlm_experimental/z2/train_colab.py \
  --minutes 30 \
  --out "$OUT" \
  --resume-best


## Show result


In [ ]:
import json
print(json.dumps(json.load(open(f"{OUT}/result.json")), indent=2, sort_keys=True))

## Optional continuation

The run cell already resumes from `$OUT/best.pt` automatically.

To force a specific checkpoint:

```python
!python /content/nanlm_experimental/z2/train_colab.py \
  --minutes 30 --out "$OUT" \
  --checkpoint-path "$OUT/best.pt"
```

To start fresh even when `best.pt` exists:

```python
!python /content/nanlm_experimental/z2/train_colab.py \
  --minutes 30 --out "$OUT" --no-resume-best
```
